# Emoji Prediction LSTM

## Loading Essential Packages

In [47]:
import numpy as np
import pandas as pd
import emoji

from keras.models import Sequential
from keras.layers import Input, TextVectorization, Dense, LSTM, SimpleRNN, Embedding
from keras.utils import to_categorical

## Loading Dataset

In [48]:
data = pd.read_csv("../../data/NLP/emoji_data.csv", header = None)
data.head()

,0,1
0,French macaroon is so tasty,4
1,work is horrible,3
2,I am upset,3
3,throw the ball,1
4,Good joke,2


In [63]:
emoji_dict = {
    0: ":red_heart:",
    1: ":baseball:",
    2: ":grinning_face_with_big_eyes:",
    3: ":disappointed_face:",
    4: ":fork_and_knife_with_plate:",
}

def label_to_emoji(label):
    return emoji.emojize(emoji_dict[int(label)])

In [50]:
X = data[0].values
Y = data[1].values

In [51]:
Y = pd.Series(Y).astype(str).str.strip()
Y = Y.replace('0v2', '0')
Y = Y.astype(int)

## Word Embeddings

In [52]:
vectorizer = TextVectorization(
    max_tokens=None,
    output_mode="int"
)

vectorizer.adapt(X)

Xtrain = vectorizer(X)

Ytrain = to_categorical(Y)

In [53]:
vocabulary = vectorizer.get_vocabulary()

vocab_size = len(vocabulary)

print("Vocab Size: ", vocab_size)

Vocab Size:  314


In [54]:
embeddings = {}

with open("../../data/NLP/glove.6B.100d.txt", "r", encoding="utf-8") as file:
    for line in file:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype = "float32")
        embeddings[word] = vector

In [55]:
embed_size = 100

embedding_matrix = np.zeros((vocab_size, embed_size))

for i, word in enumerate(vocabulary):
    if word in embeddings:
        embedding_matrix[i] = embeddings[word]

In [56]:
Y.unique()

array([4, 3, 1, 2, 0])

In [57]:
Ytrain = to_categorical(Y, num_classes=5)

## Model Creation

In [58]:
maxlen = 10

In [76]:
model = Sequential([
    Input(shape = (maxlen,)),

    Embedding(
        input_dim = vocab_size,
        output_dim = embed_size,
        weights = [embedding_matrix],
        trainable = False
    ),

    LSTM(16, return_sequences=True),
    LSTM(4),

    Dense(5, activation = "softmax")
])

In [77]:
model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [78]:
model.fit(
    Xtrain,
    Ytrain,
    epochs = 100
)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.2077 - loss: 1.6030
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3224 - loss: 1.5687
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3224 - loss: 1.5405 
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3279 - loss: 1.5219 
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3388 - loss: 1.5101
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3388 - loss: 1.4955 
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3607 - loss: 1.4807 
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3770 - loss: 1.4605
Epoch 9/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3934 - loss: 1.4388
Epoch 10/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4208 - loss: 1.4138 
Epoch 11/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4754 - loss: 1.3873
Epoch 12/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4809 - los

In [79]:
test = ["I love you", "I feel very bad", "lets eat dinner"]

Xtest = vectorizer(test)

predictions = model.predict(Xtest)

predictions = np.argmax(predictions, axis = 1)

for sentence, label in zip(test, predictions):
    print(sentence, label_to_emoji(label))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step
I love you ❤️
I feel very bad ❤️
lets eat dinner 🍽️


In [72]:
emoji_dict

{0: ':red_heart:',
 1: ':baseball:',
 2: ':grinning_face_with_big_eyes:',
 3: ':disappointed_face:',
 4: ':fork_and_knife_with_plate:'}

In [170]:
X = np.array(["I love dogs", "Our dogs eat too much meat"])
Y = np.array([0, 4])

vectorizer = TextVectorization(
    max_tokens=None,
    output_mode="int"
)

vectorizer.adapt(X)



In [171]:
vectorizer.get_vocabulary()

['',
 '[UNK]',
 np.str_('dogs'),
 np.str_('too'),
 np.str_('our'),
 np.str_('much'),
 np.str_('meat'),
 np.str_('love'),
 np.str_('i'),
 np.str_('eat')]

In [172]:
Xtrain = vectorizer(X)

In [173]:
X

array(['I love dogs', 'Our dogs eat too much meat'], dtype='<U26')

In [174]:
Xtrain

<tf.Tensor: shape=(2, 6), dtype=int64, numpy=
array([[8, 7, 2, 0, 0, 0],
       [4, 2, 9, 3, 5, 6]])>

In [175]:
Y

array([0, 4])

In [176]:
Ytrain = to_categorical(Y)
Ytrain

array([[1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1.]])

In [177]:
vocabulary = vectorizer.get_vocabulary()
vocab_size = len(vectorizer.get_vocabulary())

In [178]:
embedding_matrix = np.zeros((vocab_size, 100))

In [179]:
len(embedding_matrix)

10

In [180]:
for i, word in enumerate(vocabulary):
    if word in embeddings:
        embedding_matrix[i] = embeddings[word]

embedding_matrix

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+0

In [181]:
model = Sequential([
    Input(shape = (maxlen,)),

    Embedding(
        input_dim = vocab_size,
        output_dim = embed_size,
        weights = [embedding_matrix],
        trainable = False
    ),

    LSTM(16, return_sequences=True),
    LSTM(4),

    Dense(5, activation = "softmax")
])

In [182]:
model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [183]:
model.fit(
    Xtrain,
    Ytrain,
    epochs = 20
)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.0000e+00 - loss: 1.6133
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.0000e+00 - loss: 1.5843
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.0000e+00 - loss: 1.5568
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.0000e+00 - loss: 1.5306
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.5000 - loss: 1.5056
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.5000 - loss: 1.4819
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.5000 - loss: 1.4592
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 1.0000 - loss: 1.4376
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 1.0000 - loss: 1.4169
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 1.0000 - loss: 1.3970
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 1.0000 - loss: 1.3777
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 1.0000 - 

In [200]:
test = ["Dog loves to eat"]

Xtest = vectorizer(test)

predictions = model.predict(Xtest)

predictions = np.argmax(predictions, axis = 1)

for sentence, label in zip(test, predictions):
    print(sentence, label_to_emoji(label))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Dog loves to eat 🍽️
